# 10 -- Ambient and amplification

Step-by-step ambient analysis, HVSR and spectral amplification on different bases.

In [ ]:
import sys
sys.path.insert(0, "../src")          # run from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"   # folder with the .h5 files

In [ ]:
ds = SensorDataset(path=DATA, date_source="filename", load_mode="auto", verbose=True)
ds.verbose  = True      # internal prints, ShakerMakerResults style
ds.n_jobs   = 4
ds.parallel = True
ds.devices

In [ ]:
config = {"Fs": ds.fs, "STA":1.0, "LTA":30.0, "vent":60.0, "vmin":0.2, "vmax":2.5,
          "p":0.05, "bexp":40, "f1":0.1, "f2":25.0, "vent_seismic":False}

## Ambient, step by step (no filter inside)

In [ ]:
sig = ds.MOF00135.window("2026-05-31 15:00", "30min").signal().baseline().filter(0.1, 24.9)
amb = sig.ambient(config, component="x")
amb.sta_lta()
amb.select_windows(manual=None)
amb.taper()
amb.fft(apply_filter=False)
amb.smooth()
amb.average()
amb.sta_lta_ratio, amb.windows_signal, amb.win_ids, amb.mean_spectrum, amb.dominant_period

## One-liner (runs the missing steps lazily)

In [ ]:
sig.ambient(config, component="x").mean_spectrum

## HVSR (Nakamura)

In [ ]:
ds.MOF00135.hvsr(config=config, comp_h=("x","y"), comp_v="z", combine="geometric")

## Amplification on different bases

In [ ]:
ds.amplification(ref="MOF00135", others=["MOF00134","MNAT0031"], basis="fourier",  component="x", config=config)
ds.amplification(ref="MOF00135", others=["MOF00134","MNAT0031"], basis="response", component="x", config=config)
ds.MOF00135.amplification(basis="hvsr", config=config)